# 04 - RF Default Current Stress
Template eksperimen standar untuk MLflow tracking.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import sys

CANDIDATE_DIRS = [
    Path.cwd(),
    Path.cwd() / "nostressia-machine-learning" / "Current-Stress" / "notebooks" / "experiments",
]
for _dir in CANDIDATE_DIRS:
    if (_dir / "mlflow_utils.py").exists():
        sys.path.insert(0, str(_dir))
        break


import mlflow
import mlflow.sklearn
import pandas as pd
from mlflow.models import infer_signature
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score

from mlflow_utils import (
    RANDOM_STATE,
    configure_mlflow,
    set_seeds,
    load_current_stress_dataset,
    select_feature_set,
    split_data,
    build_preprocessor,
    evaluate_classification,
    log_classification_artifacts,
    log_run_metadata,
    train_test_dataset_frame,
    temp_artifact_dir,
)

repo_root = configure_mlflow()
set_seeds(RANDOM_STATE)


In [ ]:
# 1) Header & Config
RUN_NAME = "RF Default - Current Stress"
FEATURE_SET = "all"
REGISTERED_MODEL_NAME = "CurrentStress_RF_Default"

# 2-5) Load dataset, define target/features, split
raw_df, feature_df, y = load_current_stress_dataset(repo_root)
X = select_feature_set(feature_df, FEATURE_SET)
X_train, X_test, y_train, y_test = split_data(X, y)

# 6) Build preprocessing transformer
num_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=["number"]).columns.tolist()
preprocessor = build_preprocessor(num_cols, cat_cols)

# 7) Build model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, n_estimators=300, n_jobs=-1))
])

# 8-11) Train, evaluate, and MLflow logging
with mlflow.start_run(run_name=RUN_NAME) as run:
    dataset_payload = train_test_dataset_frame(X_train, X_test, y_train, y_test)
    run_description = (
        "RF Default - Current Stress; features=all; dataset=current_stress_v1; split=80/20; random_state=42"
    )
    log_run_metadata(
        run_description=run_description,
        tags={"features": FEATURE_SET, "model": "RF_Default"},
        params={"model_class": "CurrentStress_RF_Default", "registered_model_name": REGISTERED_MODEL_NAME, **{"n_estimators": 300}},
        dataset_df=dataset_payload,
        dataset_context="training",
    )

    
    model.fit(X_train, y_train)
    

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test) if hasattr(model, "predict_proba") else None
    metrics = evaluate_classification(y_test, y_pred, y_proba)
    mlflow.log_metrics(metrics)

    with temp_artifact_dir() as td:
        artifact_dir = Path(td)
        log_classification_artifacts(y_test, y_pred, artifact_dir, y_proba, prefix="test")
        mlflow.log_artifacts(str(artifact_dir), artifact_path="evaluation")

    signature = infer_signature(X_train, model.predict(X_train))
    mlflow.sklearn.log_model(
        sk_model=model,
        artifact_path="model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name=REGISTERED_MODEL_NAME,
    )

    print("Run ID:", run.info.run_id)
    print(pd.Series(metrics).sort_index())
